# Assignment 2 - Part 2: Hunga Tonga Analysis
------

In [ ]:
# Packages & import
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## Google Earth Images Work

In [ ]:
from pathlib import Path

data_dir = Path("sentinel-data")
for folder in sorted(data_dir.glob("S2_*")):
    print(folder.name)
    for f in sorted(folder.iterdir()):
        print("   ", f.name)

In [ ]:
data_dir = Path("sentinel-data")
wl_dir = data_dir / "White_light"

for f in sorted(wl_dir.iterdir()):
    print(f.name)

In [ ]:
def load_white_light_files(folder):
    """Returns {file_stem: image_array}, mirroring the earlier loaders."""
    folder = Path(folder)
    out = {}
    exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() in exts:
            with Image.open(path) as im:
                out[path.stem] = np.array(im.convert("RGB"))
    return out

wl = load_white_light_files(wl_dir)

print(f"{'File':30s} {'Data shape':18s} Data type")
for name, img in wl.items():
    print(f"{name:30s} {str(img.shape):18s} {img.dtype}")

In [ ]:
name = list(wl)[0]
img = wl[name]

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img)
ax.set_title(name)
plt.show()

In [ ]:
# Zoom on the bottom ~15% of the image, where the scale bar usually sits
h = img.shape[0]
y0 = int(0.85 * h)
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(img[y0:], extent=[0, img.shape[1], h, y0])
plt.show()

In [ ]:
def longest_run(row):
    """Longest run in a 1D boolean array -> (length, start, end)."""
    padded = np.concatenate([[0], row.astype(np.int8), [0]])
    edges = np.diff(padded)
    starts, ends = np.where(edges == 1)[0], np.where(edges == -1)[0]
    if len(starts) == 0:
        return 0, None, None
    i = np.argmax(ends - starts)
    return ends[i] - starts[i], starts[i], ends[i]

def measure_scale_bar(img, bottom_frac=0.15, thresh=200, min_len=20):
    """
    Search the bottom strip of the image for the longest horizontal run of
    bright pixels (the scale bar). Returns (length_px, row, x_start, x_end)
    """
    h = img.shape[0]
    y0 = int((1 - bottom_frac) * h)
    bright = img[y0:].min(axis=-1) > thresh        # all of R, G, B bright

    best = (0, None, None, None)
    for r, row in enumerate(bright):
        length, xs, xe = longest_run(row)
        if length > best[0]:
            best = (length, y0 + r, xs, xe)
    if best[0] < min_len:
        raise ValueError("No scale bar found")
    return best

In [ ]:
def show_detection(name, img, result):
    length, row, xs, xe = result
    h = img.shape[0]
    y0 = int(0.85 * h)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.imshow(img[y0:], extent=[0, img.shape[1], h, y0])
    ax.hlines(row, xs, xe, colors="red", lw=1)
    ax.set_title(f"{name}: {length} px")
    plt.show()

results = {}
for name, img in wl.items():
    results[name] = measure_scale_bar(img)
    show_detection(name, img, results[name])

In [ ]:
scale_bar_m = 600
m_per_px = scale_bar_m / L
print(f"{m_per_px:.3f} m/px, pixel area {m_per_px**2:.2f} m^2")

In [ ]:
def load_rgb(path):
    img = mpimg.imread(path).astype(float)
    if img.max() > 1.5:                    # uint8 -> 0..1
        img /= 255.0
    return img[..., :3]

def norm_diff(a, b):
    """Normalised difference (a - b) / (a + b), range -1..1."""
    return (a - b) / (a + b + 1e-10)

def colour_diffs(img):
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    return {
        "red-blue":   norm_diff(r, b),
        "red-green":  norm_diff(r, g),
        "green-blue": norm_diff(g, b),
    }

def analysis_region(img, y0, border=5):
    """True where pixels should be analysed (excludes scale bar strip and frame)."""
    roi = np.ones(img.shape[:2], bool)
    roi[y0:, :] = False
    roi[:border, :] = False
    roi[:, :border] = False
    roi[:, -border:] = False
    return roi

In [ ]:
plt.close('all')

from pathlib import Path

wl_dir = Path("sentinel-data") / "White_light"
white_images = sorted(wl_dir.glob("*.jpg"))     # list of Path objects

for i, p in enumerate(white_images):
    print(i, p.name)
img = load_rgb(white_images[1])
y0 = int(0.8 * img.shape[0])                  # or the y0 from find_scale_bar
roi = analysis_region(img, y0)
diffs = colour_diffs(img)

fig, ax = plt.subplots(ncols=3, figsize=(15, 4), tight_layout=True)
bins = np.linspace(-1, 1, 1000)
for a, (label, d) in zip(ax, diffs.items()):
    h, edges = np.histogram(d[roi], bins=bins)
    a.plot(edges[:-1], h)
    a.set_yscale("log")
    a.set_title(label)
plt.show()

In [ ]:
# INDEX = "red-blue"

In [ ]:
scale_bar_m = {
    # "stem": 600,       # label in metres, per image
}
default_bar_m = 600

def pixel_size(path, img):
    cands, y0 = find_scale_bar((img * 255).astype(np.uint8))
    if not cands:
        raise RuntimeError(f"No scale bar found in {Path(path).name}")
    L, xa, xb = cands[0]
    bar_m = scale_bar_m.get(Path(path).stem, default_bar_m)
    m_per_px = bar_m / L
    return m_per_px, y0, L

In [ ]:
plt.close('all')
white_images = sorted(p for p in wl_dir.iterdir() if p.suffix.lower() in {".jpg"})
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(10, 12), tight_layout=True)
axes = axes.ravel()
for i, ax in enumerate(axes):
    img = mpimg.imread(white_images[i])
    ax.imshow(img)

In [ ]:

wl_dir = Path("sentinel-data") / "White_light"

EXCLUDE = {"2021Dec"}

white_images = sorted(
    p for p in wl_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg"} and p.stem not in EXCLUDE
)
print([p.stem for p in white_images])      # check 2021Dec is gone
# white_images = sorted(p for p in wl_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg"})

plt.close('all')
for path in white_images:
    rgb_image = mpimg.imread(path)[:, :, :3].astype(float)   # float, so differences don't wrap
    if rgb_image.max() <= 1.0:
        rgb_image = rgb_image * 255

    red_hist = np.histogram(rgb_image[:, :, 0], bins=np.arange(0, 255))
    green_hist = np.histogram(rgb_image[:, :, 1], bins=np.arange(0, 255))
    blue_hist = np.histogram(rgb_image[:, :, 2], bins=np.arange(0, 255))

    total = np.sum(rgb_image, axis=-1)
    total[total == 0] = np.nan          # avoid divide-by-zero on pure black pixels
    diff_red_blue = (rgb_image[:, :, 0] - rgb_image[:, :, 2]) / total
    diff_red_green = (rgb_image[:, :, 0] - rgb_image[:, :, 1]) / total
    diff_green_blue = (rgb_image[:, :, 1] - rgb_image[:, :, 2]) / total

    bins = np.linspace(-1, 1, 1000)
    rb_hist = np.histogram(diff_red_blue[~np.isnan(diff_red_blue)], bins=bins)
    rg_hist = np.histogram(diff_red_green[~np.isnan(diff_red_green)], bins=bins)
    gb_hist = np.histogram(diff_green_blue[~np.isnan(diff_green_blue)], bins=bins)

    # fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(15, 8), tight_layout=True)
    # fig.suptitle(path.name)
    #
    # # Row 1: channel histograms
    # ax[0, 0].plot(red_hist[1][:-1], red_hist[0], color="red");     ax[0, 0].set_title("Red")
    # ax[0, 1].plot(green_hist[1][:-1], green_hist[0], color="green"); ax[0, 1].set_title("Green")
    # ax[0, 2].plot(blue_hist[1][:-1], blue_hist[0], color="blue");   ax[0, 2].set_title("Blue")
    #
    # # Row 2: normalised colour differences
    # ax[1, 0].plot(rb_hist[1][:-1], rb_hist[0]); ax[1, 0].set_title("(R - B) / (R+G+B)")
    # ax[1, 1].plot(rg_hist[1][:-1], rg_hist[0]); ax[1, 1].set_title("(R - G) / (R+G+B)")
    # ax[1, 2].plot(gb_hist[1][:-1], gb_hist[0]); ax[1, 2].set_title("(G - B) / (R+G+B)")
    #
    # for a in ax.ravel():
    #     a.semilogy()
    # plt.show()

In [ ]:
# rgb = mpimg.imread(white_images[2])[:, :, :3].astype(float)   # 2014Feb.jpg
# if rgb.max() <= 1.0:
#     rgb *= 255
#
# total = rgb.sum(axis=-1)
# total[total == 0] = np.nan
# idx = {
#     "R-B": (rgb[..., 0] - rgb[..., 2]) / total,
#     "R-G": (rgb[..., 0] - rgb[..., 1]) / total,
#     "G-B": (rgb[..., 1] - rgb[..., 2]) / total,
# }
#
# plt.close('all')
# fig, axes = plt.subplots(2, 3, figsize=(16, 9), tight_layout=True)
# for j, (name, d) in enumerate(idx.items()):
#     im = axes[0, j].imshow(d, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
#     axes[0, j].set_title(name); plt.colorbar(im, ax=axes[0, j], fraction=0.046)
#     h, e = np.histogram(d[~np.isnan(d)], bins=np.linspace(-0.3, 0.3, 400))
#     axes[1, j].plot(e[:-1], h); axes[1, j].semilogy()
# plt.show()

In [ ]:
for path in white_images:
    rgb = mpimg.imread(path)[:, :, :3].astype(float)
    if rgb.max() <= 1.0:
        rgb *= 255

    total = rgb.sum(axis=-1)
    total[total == 0] = np.nan
    idx = {
        "R-B": (rgb[..., 0] - rgb[..., 2]) / total,
        "R-G": (rgb[..., 0] - rgb[..., 1]) / total,
        "G-B": (rgb[..., 1] - rgb[..., 2]) / total,
    }

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), tight_layout=True)
    fig.suptitle(path.name)
    for ax, (name, d) in zip(axes, idx.items()):
        im = ax.imshow(d, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
        ax.set_title(name)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.show()
    #
    # fig, axes = plt.subplots(1, 3, figsize=(16, 9), tight_layout=True)
    # fig.suptitle(path.name)
    # for j, (name, d) in enumerate(idx.items()):
    #     im = axes[0, j].imshow(d, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
    #     axes[0, j].set_title(name)
    #     plt.colorbar(im, ax=axes[0, j], fraction=0.046)
    #     # h, e = np.histogram(d[~np.isnan(d)], bins=np.linspace(-0.3, 0.3, 400))
    #     # axes[1, j].plot(e[:-1], h)
    #     # axes[1, j].semilogy()
    #     # axes[1, j].set_xlabel(name)
    # plt.show()

In [ ]:
# RED_THRESH = 0.05    # G-B above this counts as "red"
# red_counts = {}
#
# for path in white_images:
#     rgb = mpimg.imread(path)[:, :, :3].astype(float)
#     if rgb.max() <= 1.0:
#         rgb *= 255
#
#     total = rgb.sum(axis=-1)
#     total[total == 0] = np.nan
#     gb = (rgb[..., 1] - rgb[..., 2]) / total
#
#     red = (gb > RED_THRESH) & ~np.isnan(gb)       # NaN > x is False anyway, but be explicit
#
#     # exclude the logo / scale bar strip and the frame
#     red[int(0.88 * rgb.shape[0]):, :] = False
#     red[:5, :] = False; red[:, :5] = False; red[:, -5:] = False
#
#     red_counts[path.stem] = int(red.sum())*3.79*1e-6
#
#     print(f"{path.name:20s} land area: {red_counts[path.stem]}")

In [ ]:
RED_THRESH_DEFAULT = 0.05
RED_THRESH = {
    "2014Feb": 0.03,
    "2016May": 0.01,
    "2020Nov": 0.01,
    "2022Jan": 0.15,
    "2023Oct": 0.01,
}

red_counts, red_masks, gb_maps = {}, {}, {}

for path in white_images:
    thresh = RED_THRESH.get(path.stem, RED_THRESH_DEFAULT)

    rgb = mpimg.imread(path)[:, :, :3].astype(float)
    if rgb.max() <= 1.0:
        rgb *= 255

    total = rgb.sum(axis=-1)
    total[total == 0] = np.nan
    gb = (rgb[..., 1] - rgb[..., 2]) / total

    red = (gb > thresh) & ~np.isnan(gb)

    # exclude the logo / scale bar strip and the frame
    red[int(0.88 * rgb.shape[0]):, :] = False
    red[:5, :] = False; red[:, :5] = False; red[:, -5:] = False

    gb_maps[path.stem] = gb
    red_masks[path.stem] = red
    red_counts[path.stem] = int(red.sum()) * 3.79 * 1e-6

    print(f"{path.name:20s} thresh = {thresh:.3f}   land area: {red_counts[path.stem]:.3f} km²")

# ---- plot: G-B map | G-B map with counted pixels highlighted ----
plt.close('all')
n = len(white_images)
fig, axes = plt.subplots(n, 2, figsize=(14, 6 * n), tight_layout=True, squeeze=False)

for (ax_gb, ax_hl), path in zip(axes, white_images):
    stem = path.stem
    gb, mask = gb_maps[stem], red_masks[stem]
    thresh = RED_THRESH.get(stem, RED_THRESH_DEFAULT)

    im = ax_gb.imshow(gb, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
    ax_gb.set_title(f"{stem}: G-B")
    plt.colorbar(im, ax=ax_gb, fraction=0.046)

    ax_hl.imshow(gb, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
    ax_hl.imshow(np.ma.masked_where(~mask, mask), cmap="Wistia", alpha=0.8)
    ax_hl.set_title(f"counted (G-B > {thresh:.2f}): {mask.sum()} px, {red_counts[stem]:.2f} km²")

    ax_gb.axis("off"); ax_hl.axis("off")

plt.show()

In [ ]:
areas = [red_counts[p.stem] for p in white_images]
labels = [p.name.removesuffix(".jpg") for p in white_images]

plt.close('all')
plt.figure(figsize=(10, 5))
plt.plot(areas, "o-")

plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.xlabel("Year")
plt.ylabel("Land area (km$^2$)")
plt.title("Land Area by Image")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()